---
title: Linked-Read Preprocess Summary
subtitle: A look at barcode preprocessing in raw reads
date: "9999-12-31"
edit_url: null
---
**Linked-read type**: Haplotagging (GIH)

This report describes linked-read library performance, as assessed after barcode preprocessing[^1] from the sample sequences. The values in the boxes below are averages across your samples. The reads have been modified to remove all _technical_ sequences from the R1, which includes the 4 combinatorial barcodes, the spacers between them, and the ME sequence. The data still requires thorough quality assessment, which can be achieved with e.g. [harpy qc](https://pdimens.github.io/harpy/workflows/qc/).

[^1]:
    Preprocessing ('demultiplexing') barcodes describes identifying the barcode segments and moving them into the `BX` and `VX` header tags.

In [ ]:
import altair as alt
from collections import Counter
import os
from pathlib import Path
import polars as pl
from harpy.report.components import ITable, StatsBox, image_viewer
from harpy.report.theme import palette


In [ ]:
indir = "Preprocess/reports/data"
d = {}

In [ ]:
def static_kneeplot(col: pl.Series, title: str, output_path: str):
    '''Create an Altair scatterplot as a PNG and save it to disk'''
    alt.data_transformers.disable_max_rows()
    counts = pl.DataFrame(list(Counter(col).items()), schema = ['reads', 'count'], orient = 'row')
    counts = counts.with_columns(pl.col('count') / sum(counts['count']))
    df = col.sort(descending = True).to_frame().with_row_index(name="index")
    chart = (
        alt.Chart(df)
        .mark_point()
        .encode(
            x=alt.X("index:Q", title="Barcode")
                .axis(labels=False, ticks=False, domain=False),
            y=alt.Y(f"{col.name}:Q", title = "Read Count").axis(tickMinStep=1),
            color = alt.Color(f"{col.name}:O").scale(scheme="plasma", reverse = True)
        )
        .properties(width=650, height=250)
    )
    counttable = (
        alt.Chart(counts)
        .mark_bar(align = 'center', size = 20)
        .encode(
            x = alt.X('count:Q', title = "Barcodes")
                .axis(tickMinStep = 0.1, format = ".0%", grid = False, orient = 'top')
                .scale(domain=(0, 1)),
            color = alt.Color("reads:O").scale(scheme="plasma", reverse = True),
        )
        .properties(width = 650, height = 20)
)
    concatchart = alt.vconcat(counttable, chart, title = title)
    concatchart.save(output_path, format = "png")

In [ ]:
BX_infiles = list(Path(indir).glob("*.BXstats"))

d['Sample'] = []
d['Total Reads'] = []
d['Unique Barcodes'] = []
d['Valid Barcodes'] = []
d['Corrected Barcodes'] = []
d['ME Absent'] = []

for infile in BX_infiles:
    smp = infile.stem.removesuffix('.BXstats')
    d['Sample'].append(smp)
    MEfile = os.path.join(indir, f"{smp}.MEstats")
    with open(MEfile, 'r') as f:
        total = int(f.readline().split()[-1])
        d['Total Reads'].append(total)
        d['ME Absent'].append(round(int(f.readline().split()[-1]) / total * 100, 2))
    with open(infile, 'r') as f:
        uniq = int(f.readline().split()[-1])
        d['Unique Barcodes'].append(uniq)
        d['Valid Barcodes'].append(round(int(f.readline().split()[-1]) / 2 / total * 100, 2))
        d['Corrected Barcodes'].append(round(int(f.readline().split()[-1]) / total * 100, 2))

data = pl.DataFrame(d)


In [ ]:
avg_valid = data["Valid Barcodes"].mean()
std_valid = data["Valid Barcodes"].std() or 0

avg_corrected = data["Corrected Barcodes"].mean()
std_corrected = data["Corrected Barcodes"].std() or 0

avg_me_discard = data["ME Absent"].mean()
std_me_discard = data["ME Absent"].std() or 0

(
    StatsBox()
    .add(len(d['Sample']), "Samples")
    .conditional(round(avg_valid,1), "% Valid", 75, plus_minus = round(std_valid,1))
    .add(round(avg_corrected, 5), "% Corrected", round(std_corrected, 5))
    .conditional(round(avg_me_discard, 5), "% ME Discard", 20, lower_bad = False, plus_minus = round(std_me_discard, 5))
    .render()
)

In the table below, `Valid Barcodes` is the percent of read pairs with a valid barcode. The `Corrected Barcodes` is the percent of reads pairs that had at least one unidentified barcode segment that was recovered by `Pheniqs`. The `ME Absent` column tracks the percent of read pairs that were removed due to the ME sequence not being found.

In [ ]:
ITable(data, filename = "preprocess.csv").render()

## Reads per Barcode
This is a series of kneeplots, one for each sample. The plot shows the number of reads per individual unique _valid_ barcode, sorted from highest to lowest.

In [ ]:
kneeplot_dir = os.path.join(os.path.dirname(indir), "kneeplots")
os.makedirs(kneeplot_dir, exist_ok=True)
for smp in d['Sample']:
    infile = os.path.join(indir, f"{smp}.bxcount")
    df = pl.read_csv(infile, separator = "\t", has_header = False)
    df.columns = ["bc", "count"]
    static_kneeplot(df["count"], smp, os.path.join(kneeplot_dir, f"{smp}.png"))

image_viewer("Sample", kneeplot_dir, "*.png", thing_to_select="sample")


In [ ]:
[os.remove(i) for i in Path(kneeplot_dir).glob("*.png")]
os.removedirs(kneeplot_dir)
